# AVA 云端算力工作站（Google Colab Bootstrap）

本笔记本把 AVA 的云端 GPU 推理（TTS 配音、ASR 质检、VLM 镜头意象打标）接入 Google Colab（Google AI Pro 算力）：
- **卡型建议**：仅跑 `tts` / `asr` 选 **T4 (16GB)** 即可；跑 `captions` (Qwen3-VL-8B) 请选 **L4 (24GB)** 或 **A100**。
- **两种连接模式**：
  1. `ssh`（推荐）：开启 `sshd` + `cloudflared` 隧道，本机复用 `cloud doctor/push/run/pull/logs/down --provider colab`。
  2. `drive`（零隧道兜底）：通过 Google Drive `ava_cloud/` 共享队列轮询接单，不依赖本机公网或 `cloudflared`。
- **节省 CU**：Cell 1 会把虚拟环境包缓存进 Drive；Cell 2 在空闲超时或收到本机 `cloud down` 信号时自动调用 `google.colab.runtime.unassign()`。

In [ ]:
#@title 1. 挂载 Drive 缓存、准备 venv、拉取仓库与按需下载 HuggingFace 模型
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/songhuiming2007-coder/anime-video-agent.git" #@param {type:"string"}
REPO_BRANCH = "main" #@param {type:"string"}
REPO_DIR = "/content/anime-video-agent" #@param {type:"string"}
MODELS_ROOT = "/content/models" #@param {type:"string"}
DRIVE_CACHE = "/content/drive/MyDrive/ava_cache" #@param {type:"string"}
TASK_PROFILE = "tts" #@param ["tts", "asr", "captions", "all", "none"]
FORCE_REINSTALL_VENV = False #@param {type:"boolean"}

def run(cmd, check=True):
    print(f"$ {cmd}")
    return subprocess.run(["bash", "-lc", cmd], check=check)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
Path(DRIVE_CACHE).mkdir(parents=True, exist_ok=True)
Path(MODELS_ROOT).mkdir(parents=True, exist_ok=True)

if not Path(f"{REPO_DIR}/.git").is_dir():
    run(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    run(f"git -C {REPO_DIR} fetch origin {REPO_BRANCH} --depth 1 && git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}")

venv_dir = Path("/content/venv")
py_tag = f"py{sys.version_info.major}{sys.version_info.minor}"
venv_tar = Path(DRIVE_CACHE) / f"ava_venv_{py_tag}.tar.gz"
if FORCE_REINSTALL_VENV and venv_dir.exists():
    shutil.rmtree(venv_dir)

if not (venv_dir / "bin" / "python").exists():
    if venv_tar.is_file() and not FORCE_REINSTALL_VENV:
        print(f"[*] 从 Drive 缓存恢复 venv: {venv_tar}")
        run(f"mkdir -p /content && tar -xzf {venv_tar} -C /content")
    if not (venv_dir / "bin" / "python").exists():
        run("pip install -q uv")
        run(f"uv venv --system-site-packages --python {sys.executable} {venv_dir}")
        run(
            f"cd {REPO_DIR} && uv pip install --python {venv_dir}/bin/python "
            "-e . qwen-tts funasr torch_complex 'transformers>=4.57' accelerate soundfile scipy huggingface_hub hf_transfer"
        )
        print(f"[*] 备份 venv 到 Drive 缓存以便下次秒开: {venv_tar}")
        run(f"tar -czf {venv_tar}.tmp -C /content venv && mv {venv_tar}.tmp {venv_tar}", check=False)

run(f"{venv_dir}/bin/python -c 'import torch, qwen_tts, transformers; from funasr import AutoModel; print(\"[OK] CUDA:\", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"CPU\")'")

PROFILE_MODELS = {
    "none": [],
    "tts": ["Qwen/Qwen3-TTS-12Hz-1.7B-Base", "openai/whisper-large-v3", "FunAudioLLM/SenseVoiceSmall"],
    "asr": ["openai/whisper-large-v3", "FunAudioLLM/SenseVoiceSmall"],
    "captions": ["Qwen/Qwen3-VL-8B-Instruct"],
    "all": [
        "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
        "Qwen/Qwen3-VL-8B-Instruct",
        "BAAI/bge-m3",
        "openai/whisper-large-v3",
        "FunAudioLLM/SenseVoiceSmall",
    ],
}
models_to_pull = PROFILE_MODELS[TASK_PROFILE]
if models_to_pull:
    dl_code = (
        "import os; from huggingface_hub import snapshot_download; "
        f"models = {models_to_pull!r}; "
        f"root = {MODELS_ROOT!r}; "
        "[print('[HF]', m) or snapshot_download(m, local_dir=f'{root}/{m}') for m in models]"
    )
    run(f"HF_HUB_ENABLE_HF_TRANSFER=1 {venv_dir}/bin/python -c {dl_code!r}")
print("[OK] Cell 1 完成：环境与模型已就绪。")

In [ ]:
#@title 2. 启动 AVA 服务（SSH 隧道或 Drive 共享队列，空闲或 down 自动释放 CU）
import os, re, shutil, subprocess, time
from pathlib import Path

MODE = "ssh" #@param ["ssh", "drive"]
SSH_PUBLIC_KEY = "" #@param {type:"string"}
DRIVE_QUEUE_DIR = "/content/drive/MyDrive/ava_cloud" #@param {type:"string"}
IDLE_SECONDS = 300 #@param {type:"integer"}
REPO_DIR = "/content/anime-video-agent"

def sh(cmd, check=True):
    return subprocess.run(["bash", "-lc", cmd], check=check, capture_output=True, text=True)

if MODE == "ssh":
    sh("DEBIAN_FRONTEND=noninteractive apt-get update -qq && apt-get install -y -qq openssh-server tmux rsync curl")
    if not shutil.which("cloudflared"):
        sh("curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")
    ssh_dir = Path("/root/.ssh")
    ssh_dir.mkdir(mode=0o700, exist_ok=True)
    ak = ssh_dir / "authorized_keys"
    drive_ak = Path("/content/drive/MyDrive/ava_cache/authorized_keys")
    pub = SSH_PUBLIC_KEY.strip() or (drive_ak.read_text(encoding="utf-8").strip() if drive_ak.is_file() else "")
    if not pub:
        raise RuntimeError("请填写 SSH_PUBLIC_KEY 或把本机公钥放到 /content/drive/MyDrive/ava_cache/authorized_keys")
    ak.write_text(pub + "\n", encoding="utf-8")
    ak.chmod(0o600)
    sh("mkdir -p /var/run/sshd && sed -i 's/#PubkeyAuthentication yes/PubkeyAuthentication yes/' /etc/ssh/sshd_config && service ssh restart")
    cf_log = Path("/tmp/cloudflared.log")
    if cf_log.exists():
        cf_log.unlink()
    subprocess.Popen(["cloudflared", "tunnel", "--url", "ssh://localhost:22", "--logfile", str(cf_log)], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    tunnel_host = None
    for _ in range(30):
        if cf_log.is_file():
            m = re.search(r"https://([A-Za-z0-9-]+\.trycloudflare\.com)", cf_log.read_text(errors="replace"))
            if m:
                tunnel_host = m.group(1)
                break
        time.sleep(1)
    if not tunnel_host:
        raise RuntimeError(f"未获取 trycloudflare 域名，请查看 {cf_log}")
    print("=" * 70)
    print("在本机执行以下指令接入 AVA（先选 colab，再用 connect）：")
    print(f'python -m pipeline.cloud connect "ssh -o ProxyCommand=\'cloudflared access ssh --hostname %h\' root@{tunnel_host}" --provider colab')
    print("python -m pipeline.cloud up --provider colab")
    print("=" * 70)
    import sys
    sys.path.insert(0, REPO_DIR)
    from tools import colab_worker
    colab_worker.run_ssh_sentinel(idle_seconds=int(IDLE_SECONDS), unassign_on_exit=True)
else:
    print("=" * 70)
    print(f"Colab Drive Worker 已启动，队列目录: {DRIVE_QUEUE_DIR}")
    print("本机执行: python -m pipeline.cloud use colab-drive && python -m pipeline.cloud up")
    print("=" * 70)
    import sys
    sys.path.insert(0, REPO_DIR)
    from tools import colab_worker
    colab_worker.run_drive_loop(
        Path(DRIVE_QUEUE_DIR),
        Path(REPO_DIR),
        "/content/venv/bin/python",
        Path("/content/models"),
        idle_seconds=int(IDLE_SECONDS),
        unassign_on_exit=True,
    )